In [ ]:
"""
Generate Betti Curves for H0 and H1.
Uses best configuration for each dimension.
"""

import os
import numpy as np
import pandas as pd

from gudhi.representations import BettiCurve
from scipy.ndimage import gaussian_filter1d


BASE = "RIPS"

CONFIG = {
    0: {"resolution": 100, "sigma": 2, "normalization": "l1",  "THR_MODE": "p10"},
    1: {"resolution": 150, "sigma": 2, "normalization": "none","THR_MODE": "p10"},
}

EPS = 1e-12


def read_and_save(filedir, tube):
    if tube and tube[0] != ".":
        _, ext = os.path.splitext(os.path.join(filedir, tube))
        tubenamerips = tube.split("_")[-1].split(".")[0]

        if ext != ".pdf" and tubenamerips == "Rips0":
            r0_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips0.txt")
            r1_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips1.txt")

            Rips0 = np.array(pd.read_csv(r0_path, sep=" ", header=None))
            if len(Rips0) and np.isinf(Rips0[-1, 1]):
                Rips0 = Rips0[:-1]

            Rips1 = np.array(pd.read_csv(r1_path, sep=" ", header=None))
            if len(Rips1) and np.isnan(Rips1[-1, 1]):
                Rips1[-1, 1] = 0

            return [[Rips0, Rips1], None, None]
    return []

def list_patients(root_dir):
    return [x for x in sorted(os.listdir(root_dir)) if not x.startswith(".")]

def find_rips0_file(patient_dir):
    for f in sorted(os.listdir(patient_dir)):
        if f.startswith("."):
            continue
        if f.endswith("_Rips0.txt"):
            return f
    return None

def load_group_diagrams(group_root):
    out = {}
    for patient in list_patients(group_root):
        p_dir = os.path.join(group_root, patient)
        rips0_file = find_rips0_file(p_dir)
        if rips0_file is None:
            continue
        data = read_and_save(p_dir, rips0_file)
        if not data:
            continue
        diagrams = data[0]
        if diagrams is None or len(diagrams) < 2:
            continue
        out[patient] = diagrams
    return out


def apply_persistence_threshold(pairs, thr):
    if pairs is None or len(pairs) == 0:
        return np.empty((0, 2))

    pairs = np.asarray(pairs, float)
    pers = pairs[:, 1] - pairs[:, 0]
    keep = np.isfinite(pers) & (pers >= thr)
    return pairs[keep]

def compute_global_thresholds(diag_nr, diag_r, dim_intervals, thr_mode):
    thr_dim = {}

    if thr_mode == "0":
        for d in dim_intervals:
            thr_dim[d] = 0.0
        return thr_dim

    all_diags = list(diag_nr.values()) + list(diag_r.values())

    for d in dim_intervals:
        pers_all = []

        for diagrams in all_diags:
            pairs = diagrams[d]
            if pairs is None or len(pairs) == 0:
                continue

            pairs = np.asarray(pairs, float)
            pers = pairs[:, 1] - pairs[:, 0]
            pers = pers[np.isfinite(pers)]
            pers = pers[pers > 0]

            if len(pers):
                pers_all.append(pers)

        if pers_all:
            thr_dim[d] = float(np.percentile(np.concatenate(pers_all), 10))
        else:
            thr_dim[d] = 0.0

    return thr_dim

def compute_betti(pairs, resolution):
    if pairs is None or len(pairs) == 0:
        return np.zeros(resolution)

    bc = BettiCurve(resolution=resolution)
    return bc.fit_transform([pairs])[0]

def normalize_curve(curve, mode):
    if mode == "none":
        return curve
    if mode == "l1":
        s = np.sum(np.abs(curve))
        return curve / (s + EPS)
    return curve


NR_root = os.path.join(BASE, "NonRelapse")
R_root  = os.path.join(BASE, "Relapse")

NR_diagrams = load_group_diagrams(NR_root)
R_diagrams  = load_group_diagrams(R_root)

DIM_INTERVALS = [0, 1]

for DIMENSION in DIM_INTERVALS:

    cfg = CONFIG[DIMENSION]

    thr_dim = compute_global_thresholds(
        NR_diagrams,
        R_diagrams,
        [DIMENSION],
        cfg["THR_MODE"]
    )

    thr = thr_dim[DIMENSION]

    # NON RELAPSE
    BC_NR = []
    listdirNR = sorted(NR_diagrams.keys())

    for patient in listdirNR:
        diagrams = NR_diagrams[patient]

        pairs = apply_persistence_threshold(diagrams[DIMENSION], thr)
        curve = compute_betti(pairs, cfg["resolution"])

        curve = normalize_curve(curve, cfg["normalization"])

        if cfg["sigma"] > 0:
            curve = gaussian_filter1d(curve, sigma=cfg["sigma"], mode="nearest")

        BC_NR.append(curve)

    # RELAPSE
    BC_R = []
    listdirR = sorted(R_diagrams.keys())

    for patient in listdirR:
        diagrams = R_diagrams[patient]

        pairs = apply_persistence_threshold(diagrams[DIMENSION], thr)
        curve = compute_betti(pairs, cfg["resolution"])

        curve = normalize_curve(curve, cfg["normalization"])

        if cfg["sigma"] > 0:
            curve = gaussian_filter1d(curve, sigma=cfg["sigma"], mode="nearest")

        BC_R.append(curve)

    # GUARDAR
    folder = f"BettiCurves/H{DIMENSION}"
    subfolder = os.path.join(BASE, folder)

    os.makedirs(os.path.join(subfolder, "Relapse"), exist_ok=True)
    os.makedirs(os.path.join(subfolder, "NonRelapse"), exist_ok=True)

    for i, curve in enumerate(BC_R):
        np.savetxt(os.path.join(subfolder, "Relapse", f"{listdirR[i]}.csv"), curve)

    for i, curve in enumerate(BC_NR):
        np.savetxt(os.path.join(subfolder, "NonRelapse", f"{listdirNR[i]}.csv"), curve)